# Week 5 - Dimensionality Reduction: SVD, PCA, Robust PCA, ICA & Nonlinear Embeddings

High-dimensional life-science measurements -- gene-expression matrices, imaging
stacks, multichannel physiological recordings -- are almost never as complex as
their raw dimensionality suggests. The interesting biology usually lives on a
low-dimensional surface embedded inside a much larger measurement space. This
lesson develops the linear-algebra machinery that finds that surface. You will
learn to read a **scree curve** to decide how many components a dataset truly
needs, to separate a signal into a **low-rank structure plus a sparse corruption** with robust PCA, and -- the centerpiece -- to unmix statistically
independent sources with **independent component analysis (ICA)**. The
overarching skill is not running the algorithms (a few library calls do that)
but *validating* them: every method here is first exercised against a synthetic
fixture whose answer we already know, so that when we later point it at real
biological data we can state honestly how much to trust the result.

**Reading.** Kutz, *Data-Driven Modeling & Scientific Computation*, 2nd ed.,
Chapters 15-16 (the singular value decomposition and its use in PCA/ICA). Read
those chapters for the derivations; everything below is explained in our own
terms and run against our own fixtures.

**Learning goals.**

- Connect the singular value decomposition to principal component analysis and
  read explained-variance ratios off a scree plot.
- Distinguish PCA (maximize captured variance) from ICA (maximize statistical
  independence) and know which question each one answers.
- Use a known-ground-truth fixture to *quantify* recovery quality before
  trusting a method on real data.
- Close every analysis with an explicit claim-and-limitations statement.

## Setup

We seed all random number generators and apply the course plotting style so the
figures below are deterministic and reproducible from a cold kernel.

```{admonition} Which paradigm?
:class: note
**Data-driven.** SVD and PCA make no mechanistic commitment: hand them the PBMC3k single-cell expression matrix and they return the directions of greatest variance, letting the cells' structure emerge inductively from the counts alone -- no rate law or governing equation imposed. Robust PCA (Section 2) stays inductive but asserts one structural prior -- that the matrix is low-rank plus sparse -- which is exactly what lets it quarantine gross outliers an ordinary SVD would smear. ICA stays data-driven but adds one mild prior: that your measured signals are linear mixtures of statistically independent, non-Gaussian sources -- exactly what lets it unmix overlapping fluorophores or summed neural sources where variance-maximizing PCA returns only blends. t-SNE and UMAP (Section 4) push to the far data-driven end -- nonlinear, non-parametric, non-invertible, with no loadings and no variance budget: they draw a faithful *local* picture but forfeit global metric meaning, so read them as visualization, not measurement. Reach for the linear methods when you trust your measurements but have no mechanism to deduce from, and let the known-ground-truth fixture, not faith, set how far you trust the result. Week 8 rediscovers this same PCA subspace with a linear autoencoder.
```


In [ ]:
# Colab setup: install the ddm4bio course library (+ this lesson's data deps).
# No-op when ddm4bio is already importable (e.g. the course-site build), so
# this cell is safe everywhere. It is hidden from the rendered site via the
# "remove-cell" tag, but runs when this notebook is opened in Google Colab.
try:
    import ddm4bio  # noqa: F401
except ModuleNotFoundError:
    %pip install -q "ddm4bio @ git+https://github.com/symbiont-ai/ddm4bio.git" scanpy anndata umap-learn
    import ddm4bio  # noqa: F401

In [ ]:
import numpy as np

import ddm4bio
from ddm4bio import seed_everything
from ddm4bio.viz.style import set_style

seed_everything()
set_style()

print(f"ddm4bio version: {ddm4bio.__version__}")

## 1. PCA via the SVD

Principal component analysis rotates the coordinate axes of a dataset so that
the first axis points along the direction of greatest variance, the second along
the greatest remaining variance orthogonal to the first, and so on. Numerically
this is just the singular value decomposition of the mean-centered data matrix:
the right singular vectors are the principal axes, and the squared singular
values are proportional to the variance captured along each axis.

To make PCA concrete we point it at real single-cell data. The PBMC3k dataset
is a classic 10x Genomics assay of peripheral-blood mononuclear cells from a
healthy donor: thousands of cells, each a sparse vector of gene counts. We pull
it through the course data layer, which fetches the genuine 10x matrix when it
can and otherwise returns a structurally identical synthetic single-cell matrix
so the analysis runs anywhere. The provenance line printed below tells you which
one you received; the analysis is written to work for either.

In [ ]:
from ddm4bio.datasets import get_dataset
from ddm4bio.methods.decomposition import explained_variance_ratio, pca_reduce

ds = get_dataset("pbmc3k")
payload = ds.payload

# get_dataset returns a real AnnData (with .X) or a labelled fallback dict.
if hasattr(payload, "X"):
    counts = payload.X
    labels = None
    obs = getattr(payload, "obs", None)
    if obs is not None:
        for col in ("cell_type", "cell_types", "louvain", "leiden", "bulk_labels"):
            if col in obs:
                labels = np.asarray(obs[col])
                break
else:
    counts = payload["counts"]
    labels = np.asarray(payload["labels"])

# 10x data ships as a sparse matrix; densify for the linear algebra below.
counts = np.asarray(counts.toarray() if hasattr(counts, "toarray") else counts, dtype=float)

print(f"[pbmc3k] source={ds.source}: {ds.provenance}")
groups = "unlabelled" if labels is None else f"{np.unique(labels).size} label groups"
print(f"expression matrix: {counts.shape[0]} cells x {counts.shape[1]} genes ({groups})")

Raw UMI counts are heavy-tailed and vary in sequencing depth from cell to cell,
so we apply the standard single-cell transform before any linear algebra:
normalize each cell to a common library size, then take `log1p`. We then keep
the most variable genes, which concentrates the biological signal and keeps the
full-width real matrix (tens of thousands of genes) tractable.

In [ ]:
# Library-size normalize, then log1p-compress the counts.
library = counts.sum(axis=1, keepdims=True)
library[library == 0] = 1.0
target = float(np.median(counts.sum(axis=1)))
log_counts = np.log1p(counts / library * target)

# Restrict to the top-variance genes (all of them when the matrix is already small).
n_keep = min(1000, log_counts.shape[1])
top_var = np.argsort(log_counts.var(axis=0))[::-1][:n_keep]
expr = log_counts[:, top_var]

scores = pca_reduce(expr, n_components=2)
evr = explained_variance_ratio(expr)

print(f"kept {expr.shape[1]} high-variance genes; PCA scores {scores.shape}")
print(f"leading explained-variance ratios: {np.round(evr[:5], 4)}")

Real expression data does not collapse to an exact low rank the way a clean
synthetic fixture does; instead the scree curve decays smoothly, and the "elbow"
is a judgement call about where genuine structure fades into the long tail of
biological and technical noise. We plot only the leading components, since the
tail is a near-flat noise floor.

In [ ]:
from ddm4bio.viz.plots import scree_plot

n_show = min(20, evr.size)
ax = scree_plot(evr[:n_show])
ax.figure;  # end the cell with the Figure so it renders in the notebook output

Projecting the cells onto their first two principal components gives the
familiar single-cell scatter. When the payload carries cell labels we colour by
them; the real 10x matrix ships unlabelled, so the same plot then renders as a
single-colour cloud whose structure we read from the loadings instead.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.5, 4.5))
if labels is not None:
    for g in np.unique(labels):
        sel = labels == g
        ax.scatter(scores[sel, 0], scores[sel, 1], s=12, label=str(g))
    ax.legend(title="label", fontsize=8, loc="best")
else:
    ax.scatter(scores[:, 0], scores[:, 1], s=12)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PBMC cells in principal-component space")
fig;

Finally we quantify how much of each leading component lines up with the provided
grouping, using the correlation ratio (eta): the fraction of a component's
spread that is explained by label membership, a number in `[0, 1]`.

In [ ]:
def label_separation(values, group_labels):
    """Correlation ratio (eta) of a 1-D score against categorical labels."""
    values = np.asarray(values, dtype=float)
    grand = values.mean()
    total = ((values - grand) ** 2).sum()
    if total == 0.0:
        return 0.0
    between = sum(
        int((group_labels == g).sum()) * (values[group_labels == g].mean() - grand) ** 2
        for g in np.unique(group_labels)
    )
    return float(np.sqrt(between / total))


captured = float(evr[:2].sum())
if labels is not None:
    eta1 = label_separation(scores[:, 0], labels)
    eta2 = label_separation(scores[:, 1], labels)
    print(f"QC note ({ds.source} data): the top two PCs capture {captured:.1%} of "
          f"variance; label separation is eta(PC1)={eta1:.2f}, eta(PC2)={eta2:.2f}.")
else:
    print(f"QC note ({ds.source} data): the top two PCs capture {captured:.1%} of "
          "variance; this payload ships without cell labels, so the leading axes "
          "must be read from their gene loadings, not a provided grouping.")

**QC note.** On the labelled fallback the leading component already pulls the
synthetic cell types apart (a high eta), and a handful of components carry most
of the variance -- exactly the low-dimensional structure PCA is meant to
surface. On the real unlabelled matrix we instead lean on the scree shape and
the gene loadings, and we resist over-reading components buried in the noise
tail.


## 2. Robust PCA -- low-rank signal plus sparse corruption

PCA finds the best low-rank subspace, but "best" here means least-squares, and least
squares has no defense against a few gross outliers: a handful of corrupted entries can
bend the leading singular vectors away from the real structure. Real single-cell matrices
carry exactly this kind of damage -- doublets, ambient-RNA contamination, and
dropout-inflated counts sit as *sparse gross corruption* on top of the PBMC structure from
Section 1.

Robust PCA refuses to choose between the signal and the corruption. It splits the observed
matrix into a **low-rank** part `L` (the shared structure) plus a **sparse** part `S` (the
scattered outliers), giving the corruption its own bucket instead of letting it distort `L`.

As with ICA below, real expression data has no known low-rank/sparse truth to grade against,
so we test on a fixture whose answer we built -- a rank-3 matrix plus 5% large spikes --
recover `L` and `S`, score the recovery, and contrast it with a plain rank-3 SVD of the same
matrix.


In [ ]:
from ddm4bio.methods.decomposition import rpca, svd_lowrank
from ddm4bio.methods.validation import reconstruction_error

# A fixture whose answer we know: a rank-3 signal plus 5% large sparse spikes.
rng = np.random.default_rng(0)
n_rows, n_cols, rank_true = 200, 120, 3
L_true = rng.standard_normal((n_rows, rank_true)) @ rng.standard_normal((rank_true, n_cols))
amp = 5.0 * np.abs(L_true).mean()
corrupt = rng.random((n_rows, n_cols)) < 0.05                 # which entries are corrupted
S_true = np.where(corrupt, amp * rng.choice([-1.0, 1.0], size=(n_rows, n_cols)), 0.0)
X_obs = L_true + S_true

# Robust PCA splits the observed matrix into low-rank L + sparse S.
L_hat, S_hat = rpca(X_obs)

rel_L = reconstruction_error(L_true, L_hat, kind="rel_l2")
svals = np.linalg.svd(L_hat, compute_uv=False)
eff_rank = int((svals > 1e-6 * svals[0]).sum())
found = np.abs(S_hat) > 0.5 * amp
tp = int((found & corrupt).sum())
fp = int((found & ~corrupt).sum())
fn = int((~found & corrupt).sum())
precision, recall = tp / (tp + fp), tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

# Contrast: a plain rank-3 SVD has to absorb the spikes into its factors.
U, s3, Vt = svd_lowrank(X_obs, rank_true)
rel_L_svd = reconstruction_error(L_true, (U * s3) @ Vt, kind="rel_l2")

print(f"corrupted entries : {int(corrupt.sum())} of {n_rows * n_cols} ({100 * corrupt.mean():.1f}%)")
print(f"robust PCA        : rel. L error {rel_L:.1e} | recovered rank {eff_rank} | spike F1 {f1:.2f}")
print(f"plain SVD (rank 3): rel. L error {rel_L_svd:.2f}")


In [ ]:
import matplotlib.pyplot as plt

vmax = np.abs(X_obs).max()
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), constrained_layout=True)
panels = [
    (X_obs, "Observed  X = L + S"),
    (L_true, "True low-rank  L"),
    (L_hat, f"Recovered  L_hat  (err {rel_L:.0e})"),
    (S_hat, f"Recovered sparse  S_hat  (F1 {f1:.2f})"),
]
for ax, (mat, title) in zip(axes, panels):
    im = ax.imshow(mat, cmap="coolwarm", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.85, label="value")
plt.show()


The split is exact where it counts: robust PCA recovers the true low-rank part to a relative
error of **1.2e-07** at the correct rank 3 and flags every injected spike (F1 **1.00**),
while a plain rank-3 SVD of the same matrix -- forced to explain the outliers with its
factors -- lands **17%** off the truth. That extra low-rank-plus-sparse *assumption* is the
whole difference: robust PCA still learns `L` and `S` from the data alone, but by asserting a
little more structure than bare PCA it can quarantine corruption that would otherwise bend the
components. It sits one rung up this week's ladder of asserted structure -- PCA (low-rank) ->
robust PCA (low-rank + sparse) -> ICA (independence) -- and, as always, the synthetic score is
what licenses trusting it on messy biology.


## 3. ICA with known ground truth

PCA asks "which directions carry the most variance?" ICA asks a different and
often more biologically relevant question: "which underlying signals are
*statistically independent*?" When several independent processes are linearly
mixed at each sensor -- think of overlapping fluorophores in an imaging channel,
or independent neural sources summed at a scalp electrode -- PCA will happily
find high-variance directions, but those directions are generally *mixtures*.
ICA is designed to invert the mixing and hand back the original sources.

The only honest way to trust an unmixing algorithm is to test it on data whose
sources we already know. The fixture below generates three independent,
non-Gaussian sources (ICA relies on non-Gaussianity), linearly mixes them
through a random well-conditioned matrix, and returns both the true sources and
the observed mixtures.

In [ ]:
from ddm4bio.datasets.synthetic import make_mixed_sources

m = make_mixed_sources(3, 2000, seed=0)

print(f"True sources:   {m.sources.shape} (sources x samples)")
print(f"Mixing matrix:  {m.mixing.shape}")
print(f"Observations:   {m.observations.shape} (channels x samples)")

First, look at what we are given versus what we want to recover. The top row is
the ground truth (three cleanly separated waveforms); the bottom row is what a
sensor actually records -- three tangled mixtures in which no individual source
is visible.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(11, 4.5), sharex=True)
window = slice(0, 400)  # show a short window so the waveforms are legible

for j in range(3):
    axes[0, j].plot(m.sources[j, window])
    axes[0, j].set_title(f"True source {j + 1}")
    axes[1, j].plot(m.observations[j, window])
    axes[1, j].set_title(f"Mixed observation {j + 1}")
axes[0, 0].set_ylabel("True")
axes[1, 0].set_ylabel("Observed")
fig.suptitle("Blind source separation: ground truth (top) vs. mixtures (bottom)")
fig;

Now unmix. `ica_unmix` runs FastICA on the observations and returns one
estimated source per row. We then score the recovery with
`source_recovery_score`, which optimally matches each estimated source to a true
source and reports the mean absolute correlation across the matched pairs -- a
number in `[0, 1]` where 1 means perfect recovery up to the sign and ordering
ambiguities that are inherent to ICA.

In [ ]:
from ddm4bio.methods.decomposition import ica_unmix
from ddm4bio.methods.validation import source_recovery_score

est = ica_unmix(m.observations, 3, seed=0)
score = source_recovery_score(m.sources, est)

print(f"Estimated sources: {est.shape}")
print(f"Source-recovery score: {score:.4f}")

A score near 0.999 tells us the unmixing is essentially exact on this fixture.
The plot below confirms it visually: each recovered source is overlaid on its
matched true source. ICA does not preserve the *sign* or *order* of sources, so
before plotting we align each estimate to its best-matching truth by correlation
sign -- a cosmetic fix that changes nothing about the recovery quality.

In [ ]:
from scipy.optimize import linear_sum_assignment

# Optimally match estimated sources to true sources by absolute correlation.
n_src = m.sources.shape[0]
corr = np.zeros((n_src, n_src))
for i in range(n_src):
    for j in range(n_src):
        a = m.sources[i] - m.sources[i].mean()
        b = est[j] - est[j].mean()
        corr[i, j] = np.corrcoef(a, b)[0, 1]

true_idx, est_idx = linear_sum_assignment(-np.abs(corr))

fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharex=True)
for panel, (ti, ei) in enumerate(zip(true_idx, est_idx)):
    sign = np.sign(corr[ti, ei])  # flip estimate to match the truth's sign
    axes[panel].plot(m.sources[ti, window], label="true", linewidth=2)
    axes[panel].plot(sign * est[ei, window], label="recovered",
                     linewidth=1, linestyle="--")
    axes[panel].set_title(f"Source {panel + 1}")
    axes[panel].legend(loc="upper right", fontsize=8)
fig.suptitle("Recovered sources overlaid on ground truth")
fig;

**Why this ordering matters.** We validated recovery on a fixture with a *known*
answer before ever touching real biological data. This is the non-negotiable
course rule: a method that cannot recover known synthetic sources has no business
being trusted on messy experimental measurements, where there is no ground truth
to check against. The synthetic score is your license to proceed -- or your
warning to stop.


## 4. Nonlinear neighbor embeddings: t-SNE and UMAP

Everything so far has been *linear*. PCA, robust PCA, and ICA each factor the data through
matrix multiplications, and each returns something you can inspect and invert -- loadings, an
explained-variance budget, a map you can run forwards and backwards. The figures single-cell
biology actually publishes are almost never these. They are **t-SNE** (van der Maaten &
Hinton, 2008) and **UMAP** (McInnes, Healy & Melville, 2018) embeddings: *nonlinear
neighbor-embedding* methods that arrange a 2-D picture so each cell stays near the same
neighbors it had in high dimensions.

These are **post-textbook** tools. The reference textbook takes dimensionality reduction from
the SVD through autoencoders, but t-SNE (2008) and UMAP (2018) arrived later, from the
machine-learning and bioinformatics communities. We teach them because they are the field's
workhorse -- and because they are the *most misread* methods in biology, which is exactly why
this course covers them with the caveats attached.

The standard pipeline is **PCA first, then embed**: reduce to about 50 principal components
(denoising, and shrinking tens of thousands of genes to a tractable matrix), then run
t-SNE/UMAP on those PCs. On the model-driven <-> data-driven ladder these sit at the far
data-driven end -- nonlinear, non-parametric, non-invertible. They make a *picture*, not a
measurement.


In [ ]:
import time
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import umap

# Standard pipeline: PCA to 50 components on the log-normalized top-variance genes.
pcs = pca_reduce(expr, n_components=50)
pca2 = pcs[:, :2]
print(f"PCA variance kept -- top-2 PCs: {evr[:2].sum():.1%}, top-50 PCs: {evr[:50].sum():.1%}")

# Two honest color keys for UNLABELED data: a marker gene, and k-means clusters on the PCs.
var_names = np.asarray(getattr(payload, "var_names", []), dtype=str)
hit = np.where(np.char.upper(var_names) == "LYZ")[0] if var_names.size else np.array([], dtype=int)
if hit.size:
    marker_name, marker = "LYZ", log_counts[:, int(hit[0])]   # a monocyte lineage marker
else:
    j = int(np.argmax(log_counts.var(axis=0))); marker_name, marker = f"gene {j}", log_counts[:, j]
clusters = KMeans(n_clusters=8, random_state=0, n_init=10).fit_predict(pcs)
print(f"k-means cluster sizes on the 50 PCs: {np.bincount(clusters).tolist()}")

# The three layouts, computed on the 50 PCs, seeds pinned for a reproducible figure.
t = time.perf_counter()
emb_tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(pcs)
t_tsne = time.perf_counter() - t
t = time.perf_counter()
emb_umap = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(pcs)
t_umap = time.perf_counter() - t
print(f"t-SNE {t_tsne:.1f}s | UMAP {t_umap:.1f}s  (UMAP's first call includes a one-time ~10-15s compile)")


# Do the embeddings actually help? Score LOCAL structure: how often a point's 2-D neighbors
# share its (50-D-defined) k-means label, and the silhouette of those labels in each layout.
def knn_purity(Y, lab, k=15):
    nbr = NearestNeighbors(n_neighbors=k + 1).fit(Y).kneighbors(Y, return_distance=False)[:, 1:]
    return float(np.mean(lab[nbr] == lab[:, None]))


for name, Y in [("PCA(2)", pca2), ("t-SNE", emb_tsne), ("UMAP", emb_umap)]:
    print(f"  {name:7} local kNN-purity {knn_purity(Y, clusters):.3f} | "
          f"silhouette {silhouette_score(Y, clusters):.3f}")

In [ ]:
import matplotlib.pyplot as plt

layouts = [("PCA (2 comps)", pca2), ("t-SNE", emb_tsne), ("UMAP", emb_umap)]
fig, axes = plt.subplots(2, 3, figsize=(13.5, 8), constrained_layout=True)
for col, (title, Y) in enumerate(layouts):
    s_marker = axes[0, col].scatter(Y[:, 0], Y[:, 1], c=marker, cmap="viridis", s=5)
    axes[0, col].set_title(title, fontsize=11)
    axes[1, col].scatter(Y[:, 0], Y[:, 1], c=clusters, cmap="tab10", s=5)
    for ax in (axes[0, col], axes[1, col]):
        ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_ylabel(f"colored by {marker_name}\n(marker expression)", fontsize=10)
axes[1, 0].set_ylabel("colored by k-means\ncluster on the PCs", fontsize=10)
fig.colorbar(s_marker, ax=axes[0, :], shrink=0.7, label=f"{marker_name} expression")
fig.suptitle(f"{pcs.shape[0]} pbmc3k cells -- linear PCA vs nonlinear embeddings (seed 0)", fontsize=12)
plt.show()


The nonlinear layouts earn their keep on **local** structure. Cells that share a lineage
marker (top row) collapse into tight, well-separated islands in t-SNE and UMAP, and the
k-means clusters (bottom row) resolve into discrete blobs -- exactly what makes these plots
useful for spotting cell populations. The improvement is explicitly *local*: neighbor purity
climbs from about 0.79 in the flat PCA(2) projection to about 0.91 in both embeddings, because
a 2-D linear projection keeps under a fifth of the variance and smears populations that the
50-D structure separates.

That local faithfulness is the *only* thing these plots earn you. Read the same figure
skeptically:

- **Between-cluster distance is not biological distance.** The gaps and blob shapes differ
  between the t-SNE and UMAP panels for the *same* cells; neither is a distance you can measure.
- **Cluster size and density are not to scale** -- a bigger or denser blob does not mean a more
  heterogeneous population.
- **The layout is stochastic.** We pinned `random_state=0`; `perplexity` (t-SNE) and
  `n_neighbors` (UMAP) are knobs that change the picture.
- **Small islands can be artifacts.** The k-means run above produced a cluster of only a handful
  of cells; such specks must be *verified* against marker genes, not trusted because the plot
  drew them apart.

The next block makes the first two of these concrete on a fixture whose true geometry we set.


### The plot can lie: what these embeddings distort

The course rule is that a method earns trust by recovering a *known* answer. So we build a
fixture whose geometry we set by construction -- three Gaussian clusters in 50-D: two tight
ones close together (A and B) and one diffuse cluster far away (C) -- and check what each
layout does to distances and sizes we can measure exactly.


In [ ]:
from sklearn.decomposition import PCA

rng = np.random.default_rng(0)
D = 50
cA, cB, cC = np.zeros(D), np.zeros(D), np.zeros(D)
cB[0] = 20.0       # B sits close to A
cC[1] = 200.0      # C sits ~10x farther away
XA = rng.normal(cA, 1.0, (300, D))    # tight
XB = rng.normal(cB, 1.0, (300, D))    # tight
XC = rng.normal(cC, 8.0, (300, D))    # diffuse: ~8x the spread
Xg = PCA(n_components=50, random_state=0).fit_transform(np.vstack([XA, XB, XC]))
lab = np.array([0] * 300 + [1] * 300 + [2] * 300)


def geometry(Y):
    ctr = {g: Y[lab == g].mean(0) for g in (0, 1, 2)}
    rad = {g: float(np.linalg.norm(Y[lab == g] - ctr[g], axis=1).mean()) for g in (0, 1, 2)}
    dist_ratio = np.linalg.norm(ctr[0] - ctr[2]) / np.linalg.norm(ctr[0] - ctr[1])
    return dist_ratio, rad[2] / rad[0]


emb = {
    "PCA (2 comps)": Xg[:, :2],
    "t-SNE": TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(Xg),
    "UMAP": umap.UMAP(n_components=2, random_state=0).fit_transform(Xg),
}
true_d, true_s = geometry(Xg)
print(f"TRUE geometry (in 50-D): A-C is {true_d:.0f}x farther than A-B; C is {true_s:.0f}x more spread")
for name, Y in emb.items():
    dr, sr = geometry(Y)
    print(f"  {name:14} drawn distance-ratio {dr:4.1f} (true {true_d:.0f}) | "
          f"size-ratio {sr:4.1f} (true {true_s:.0f})")

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, (name, Y) in zip(axes, emb.items()):
    ax.scatter(Y[:, 0], Y[:, 1], c=lab, cmap="Set1", s=6)
    ax.set_title(name, fontsize=11); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Same three clusters, known geometry -- only linear PCA keeps A-C far and C large",
             fontsize=12)
plt.show()


The numbers are stark: a distance the data sets at roughly 10:1 is drawn by both
t-SNE and UMAP down toward 1:1, and a cluster that is genuinely several times more spread out is
rendered about the same size as its tight neighbors (the exact compression is itself
seed- and version-dependent -- another reason not to read it as a measurement). The *linear*
PCA projection alone keeps the true proportions -- because
it is a metric projection, not a neighbor-optimized picture.

That is the whole trade. On the real pbmc3k cells above, t-SNE and UMAP gave you cleaner
*local* islands than PCA(2) could; here they destroy the *global* geometry PCA preserves. The
layout is also arbitrary up to a seed -- re-run with a different `random_state` and the global
arrangement reshuffles while each point keeps roughly the same near neighbors. So read these
plots for **local** structure -- which cells group with which -- and never read a global
measurement off them: an inter-cluster distance, a cluster size, a density. They are the far,
purely inductive end of the week's methods: maximally flexible, and for exactly that reason the
least safe to read quantitatively.


## 5. Interpretation

Every ddm4bio analysis closes with an explicit interpretation block: a single
claim, stated with the evidence that backs it, and a list of named
limitations. This forces us to write down not just *what* we found but *how much*
we should believe it and *where* it could break.

In [ ]:
from ddm4bio.interpret import interpretation_block, show_interpretation

block = interpretation_block(
    claim="FastICA recovers the three independent sources from their linear "
          "mixtures with essentially perfect fidelity on this fixture.",
    limitations_list=[
        f"Result is on a synthetic fixture (score={score:.3f}); real data are "
        "noisier and only approximately linear mixtures.",
        "Sources were constructed to be non-Gaussian and independent -- the "
        "exact assumptions ICA needs; violate them and recovery degrades.",
        "The mixing matrix was well-conditioned by construction; near-singular "
        "mixing would make the inverse problem ill-posed.",
        "Sign and ordering of recovered sources are arbitrary and must be "
        "resolved by external reference, not by ICA itself.",
    ],
)
show_interpretation(block)

## Exercises

Your graded work for this week is **Problem Set 5 (PS5)**, distributed and
auto-graded through GitHub Classroom. This lesson reads a scree curve and
*eyeballs* where genuine structure fades into the noise tail. PS5 replaces that
judgement call with a principled significance test. Building on this lesson, it
asks you to:

- Build **Horn's parallel analysis**: permute each feature (gene) column
  independently to construct a rank-matched *noise* null of the eigenvalue
  spectrum, then keep only the leading principal components whose real eigenvalue
  beats the null -- recovering the latent dimensionality with a statistical test,
  not an eyeballed elbow.
- Validate it against synthetic matrices of **known planted rank**: show the test
  recovers the injected number of components exactly across a noise sweep, and
  degrades honestly only at extreme SNR.
- Contrast parallel analysis against the naive analytic **Marchenko-Pastur edge**,
  and show why the analytic shortcut over-counts on real, non-Gaussian expression
  data (real PBMC3k) even though the two rules agree on clean Gaussian noise.
- Write an interpretation block for the result using
  `ddm4bio.interpret.interpretation_block`.

Refer to the [PS5 repository README](https://github.com/symbiont-ai/ddm4bio/tree/main/problem_sets/ps5_dimreduction) for the submission and auto-grading details.